# Camada Ouro — Modelo 3: Previsão de Próximo Trecho

**Contexto:** Este notebook constrói um motor de recomendação usando Machine Learning (LightGBM) para prever qual será o próximo destino comprado por um cliente, com base no seu histórico de navegação e no cluster de comportamento a que pertence.

> **Dependências:** Requer a execução prévia de `01_Camada_Ouro_EDA.ipynb` e `02_Camada_Ouro_Segmentacao.ipynb`.
>
> **Inputs:**
> - `prata/clickbus_treino.parquet` (Dados históricos para treino)
> - `prata/clickbus_val.parquet` (Dados futuros para validação do target)
> - `ouro/df_cliente_clusterizado.parquet` (Matriz RFM e personas)
>
> **Outputs:** `ouro/lightgbm_proximo_trecho.pkl` (Modelo final) e `ouro/df_predicoes_finais.parquet`

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, top_k_accuracy_score, accuracy_score
import lightgbm as lgb
import shap
import time

!git clone https://github.com/vsmacedo-datafinance/Challenge_ClickBus_FIAP_2025.git 2>/dev/null
sys.path.append('/content/Challenge_ClickBus_FIAP_2025')

drive.mount('/content/drive')
sns.set_theme(style='whitegrid')

DRIVE_BASE = "/content/drive/MyDrive/Portifólio DS Vini/Challenge_ClickBus_2025/data"

COLUNA_ALVO = 'trecho_ida'
RANDOM_STATE = 42

In [ ]:
df_treino = pd.read_parquet(f"{DRIVE_BASE}/prata/clickbus_treino.parquet")
df_val = pd.read_parquet(f"{DRIVE_BASE}/prata/clickbus_val.parquet")
df_cliente = pd.read_parquet(f"{DRIVE_BASE}/ouro/df_cliente_clusterizado.parquet")

In [ ]:
df_cliente.info()

In [ ]:
df_cliente.head()

In [ ]:
limiar_uma_compra = np.log1p(1) #  1 compra = np.log1p(1) = 0.693147
df_clientes_frequentes = df_cliente[df_cliente['total_compras_log'] > limiar_uma_compra].copy()

print(f"Base Original (Todos os clientes): {len(df_cliente)}")
print(f"Base Retidos (Alvo do modelo): {len(df_clientes_frequentes)}")
print(f"Redução de ruído: {len(df_cliente) - len(df_clientes_frequentes)} clientes one-shot descartados.")

### Filtro de Clientes "One-Shot"

> **Premissa Analítica:** Não faz sentido tentar prever matematicamente o "próximo" destino de utilizadores que realizaram apenas 1 compra na história (One-shot users), pois não há padrão de recorrência estabelecido.

Nesta etapa, aplicamos um corte lógico (`total_compras_log > np.log1p(1)`). O modelo irá focar-se exclusivamente nos clientes **retidos/frequentes**. Isto descarta mais de 266 mil registos ruidosos, garantindo que o algoritmo aprende padrões reais de fidelidade.

In [ ]:
df_treino.info()

Premissa: Não faz sentido prever o "próximo" destino de quem comprou apenas 1 vez (one-shot).


In [ ]:
ultimo_trecho_treino = (
    df_treino.sort_values(by=['id_cliente', 'data_compra'])
    .drop_duplicates(subset=['id_cliente'], keep='last')
    [['id_cliente', COLUNA_ALVO]]
    .rename(columns={COLUNA_ALVO: 'ultimo_trecho'})
)

# criação do Target: 1º trecho comprado pelo cliente na base de validação
primeira_viagem_val = (
    df_val.sort_values(by=['id_cliente', 'data_compra'])
    .drop_duplicates(subset=['id_cliente'], keep='first')
    [['id_cliente', COLUNA_ALVO]]
    .rename(columns={COLUNA_ALVO: 'target_trecho'})
)

df_model = df_clientes_frequentes.merge(ultimo_trecho_treino, on='id_cliente', how='inner')
df_model = df_model.merge(primeira_viagem_val, on='id_cliente', how='inner')

print(f"{len(df_model)} clientes retidos para modelagem.")

Para evitar o vazamento de dados do futuro (*Data Leakage*), a construção do nosso *target* (aquilo que queremos prever) segue uma lógica temporal rigorosa:
1. **Histórico (`ultimo_trecho`):** Extraímos a última viagem realizada no limite de tempo do `df_treino`.
2. **O Futuro (`target_trecho`):** Extraímos a **primeira viagem** efetivamente realizada no `df_val`.

Assim, simulamos o cenário real de produção: o modelo apenas olha para o passado para tentar adivinhar o primeiro movimento do cliente no mês seguinte.

In [ ]:
# foco nos 10 principais trechos
top_10_trechos = df_model['target_trecho'].value_counts().nlargest(10).index.tolist()

print("Top 10 Trechos Alvo:")
for i, trecho in enumerate(top_10_trechos, 1):
    print(f"  {i}. {trecho}")

df_model_top10 = df_model[df_model['target_trecho'].isin(top_10_trechos)].copy()
print(f"\nVolume de dados após filtro Top 10: {len(df_model_top10)} registros aptos para treino.")

In [ ]:
le_target = LabelEncoder()
df_model_top10['target_encoded'] = le_target.fit_transform(df_model_top10['target_trecho'])

cols_categoricas = ['ultimo_trecho']
for col in cols_categoricas:
    df_model_top10[col] = df_model_top10[col].astype('category')

df_model_top10['cluster'] = df_model_top10['cluster'].astype('category') # transformar em variável categórica
cols_categoricas.append('cluster')

features_drop = ['id_cliente', 'target_trecho', 'target_encoded']
X = df_model_top10.drop(columns=features_drop)
y = df_model_top10['target_encoded']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Shape Treino: {X_train.shape} | Shape Validação Interna: {X_val.shape}")

In [ ]:
lgb_estimator = lgb.LGBMClassifier(
    objective='multiclass',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_distributions = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [5, 7, 10, -1],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

search = RandomizedSearchCV(
    estimator=lgb_estimator,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='accuracy',
    cv=3,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.time()

search.fit(
    X_train,
    y_train,
    categorical_feature=cols_categoricas,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
)

tempo_total = time.time() - start_time
modelo_final = search.best_estimator_

print(f"\n Otimização concluída em {tempo_total:.1f} segundos.")
print(f"Melhores hiperparâmetros:\n{search.best_params_}")

y_pred = modelo_final.predict(X_val)
print("\n" + "="*60)
print("RELATÓRIO DE CLASSIFICAÇÃO (Validação Interna)")
print("="*60)
print(classification_report(y_val, y_pred, target_names=le_target.classes_))

### Modelação Multiclasse com LightGBM e Otimização

Optámos pelo algoritmo **LightGBM (LGBMClassifier)** por duas razões arquiteturais fortes:
1. **Tratamento Nativo de Categorias:** Este algoritmo lida excecionalmente bem com variáveis puramente categóricas (como o `cluster` e o `ultimo_trecho`) através da sua estrutura em árvore, dispensando técnicas pesadas como o *One-Hot Encoding*.
2. **Desempenho Multiclasse:** Sendo o nosso desafio prever entre os Top 10 destinos, o LightGBM é muito mais eficiente em alocação de memória e tempo de treino face a outras alternativas baseadas em *Boosting*.

Utilizamos o `RandomizedSearchCV` com `class_weight='balanced'` para garantir que o modelo não fica enviesado a prever apenas a rota mais popular.

In [ ]:
y_probs = modelo_final.predict_proba(X_val)

top3_acc = top_k_accuracy_score(y_val, y_probs, k=3)

print("="*60)
print(f"PERFORMANCE DE RECOMENDAÇÃO (TOP-3 ACCURACY)")
print("="*60)
print(f"O modelo acertou o destino exato (Top-1) em: {accuracy_score(y_val, y_pred)*100:.1f}% das vezes.")
print(f"O destino real estava entre as 3 principais recomendações em: {top3_acc*100:.1f}% das vezes.")
print("="*60)
print("Conclusão de Negócio: Se mostrarmos um carrossel com 3 destinos para estes clientes,")
print(f"temos {top3_acc*100:.1f}% de chance de acertar exatamente para onde eles querem ir!")